# 02 — Driver vs. Non-Driver Gene Labeling Pipeline

Builds the final driver/non-driver gene label set:

1. Protein-coding gene universe (GENCODE, `gene_type == "protein_coding"` only)
2. Driver genes (COSMIC CGC) as positives
3. Negative candidates excluded from **NCG + CGC + IntOGen + Bailey et al. 2018** — not just CGC
4. Mutation-frequency filter (excludes genes frequently mutated in any TCGA cancer type)
5. Disease-pathway filter (Reactome, restricted to descendants of the top-level *Disease* pathway)
6. **Length- and expression- (optionally mutation-rate-) matched negative sampling**, so negatives
   aren't just "whatever survived the filters" but are matched to the positives' covariate
   distribution

This notebook is a runnable, cell-by-cell version of `scripts/run_driver_labeling_pipeline.py` —
it imports the same `src/` modules, so there's no logic duplicated between the notebook and the
package.

**Requires** (see `config.py` and the README for expected paths):
- GENCODE GTF annotation
- COSMIC Cancer Gene Census TSV
- `data/mutation_frequency_tcga.csv` (produced by notebook 01)
- Reactome `ReactomePathways.gmt` + pathway relations CSV
- NCG / IntOGen / Bailey driver lists (optional — skipped if absent)
- Gene median expression CSV (required only if matched negative sampling is enabled)


In [1]:
import sys
import os
import random
from pathlib import Path

# Notebooks live in notebooks/, but config.py's paths (e.g. "data/...") are
# relative to the repo root — chdir there so those paths resolve correctly
# regardless of where Jupyter's working directory starts out.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd

import config
from src.gene_universe import build_protein_coding_gene_universe
from src.pathway_filter import build_disease_pathway_genes
from src.driver_labeling import (
    apply_mutation_frequency_filter,
    apply_pathway_filter,
    build_gene_labels,
    load_driver_genes,
)
from src.negative_sampling import (
    build_driver_exclusion_set,
    compute_gene_lengths,
    load_gene_expression,
    load_gene_mutation_rate,
    match_negatives,
    summarize_matching_quality,
    build_covariate_table,
)

In [2]:
random.seed(config.RANDOM_SEED)

config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"USE_MUTATION_FILTER: {config.USE_MUTATION_FILTER}")
print(f"USE_PATHWAY_FILTER: {config.USE_PATHWAY_FILTER}")
print(f"USE_MATCHED_NEGATIVE_SAMPLING: {config.USE_MATCHED_NEGATIVE_SAMPLING}")
print(f"NEGATIVES_PER_POSITIVE: {config.NEGATIVES_PER_POSITIVE}")
print(f"MATCH_ON_MUTATION_RATE: {config.MATCH_ON_MUTATION_RATE}")

USE_MUTATION_FILTER: True
USE_PATHWAY_FILTER: True
USE_MATCHED_NEGATIVE_SAMPLING: True
NEGATIVES_PER_POSITIVE: 1
MATCH_ON_MUTATION_RATE: False


## 1. Protein-coding gene universe

Parses the GENCODE GTF and keeps only `gene_type == "protein_coding"` entries — excluding
lncRNAs, pseudogenes, miRNAs, and other non-coding biotypes that would otherwise inflate the
gene universe (~77,000 -> ~20,000 genes).

In [3]:
# all_genes = build_protein_coding_gene_universe(config.GENCODE_GTF_FILE)
all_genes = build_protein_coding_gene_universe("../../data/gencode.v49.basic.annotation.gtf")

pd.DataFrame(sorted(all_genes), columns=["gene_name"]).to_csv(
    config.GENCODE_GENES_FILE, index=False
)

print(f"[GENE UNIVERSE] {len(all_genes)} protein-coding genes")

[GENE UNIVERSE] 20070 protein-coding genes


In [4]:
config.GENCODE_GTF_FILE

PosixPath('../../data/gencode.v49.basic.annotation.gtf')

## 2. Driver genes and the expanded exclusion set

Positives are defined by the COSMIC Cancer Gene Census, as before. But a gene is only eligible
to be sampled as a **negative** if it is absent from *all four* driver references (NCG, CGC,
IntOGen, Bailey et al. 2018) — a gene with driver evidence anywhere shouldn't be usable as a
"confirmed non-driver," even if it isn't in CGC specifically.

In [5]:
driver_genes = load_driver_genes("../../data/Census_allWed.tsv")
print(f"[DRIVERS] {len(driver_genes)} driver genes (CGC)")

driver_exclusion_set = build_driver_exclusion_set(
    ncg_file=config.NCG_FILE if config.NCG_FILE.exists() else None,
    cgc_file=config.CGC_CENSUS_FILE if config.CGC_CENSUS_FILE.exists() else None,
    intogen_file=config.INTOGEN_FILE if config.INTOGEN_FILE.exists() else None,
    bailey_file=config.BAILEY_FILE if config.BAILEY_FILE.exists() else None,
)

non_drivers = all_genes - driver_exclusion_set
print(f"[NON-DRIVERS] {len(non_drivers)} raw non-driver candidates "
      "(excluded from NCG/CGC/IntOGen/Bailey)")

[DRIVERS] 763 driver genes (CGC)
[NCG] skipped (no file provided)
[CGC] 763 genes
[IntOGen] skipped (no file provided)
[Bailey et al.] skipped (no file provided)
[DRIVER EXCLUSION SET] 763 unique genes across all sources
[NON-DRIVERS] 19326 raw non-driver candidates (excluded from NCG/CGC/IntOGen/Bailey)


## 3. Mutation-frequency filter

Excludes any candidate with mutation frequency >= `config.MUTATION_FREQUENCY_THRESHOLD` in any
TCGA cancer type. Genes never observed mutated at all are correctly treated as 0% frequency
(kept), not dropped by omission.

In [6]:
if config.USE_MUTATION_FILTER:
    before = len(non_drivers)
    non_drivers = apply_mutation_frequency_filter(
        non_drivers,
        config.MUTATION_FREQUENCY_FILE,
        threshold=config.MUTATION_FREQUENCY_THRESHOLD,
    )
    print(f"[MUTATION FILTER] {len(non_drivers)} remaining (-{before - len(non_drivers)})")

[MUTATION FILTER] 2491 remaining (-16835)


## 4. Disease-pathway filter

Excludes candidates belonging to a Reactome pathway that is a descendant of the top-level
*Disease* pathway (`R-HSA-1643685`), restricting the exclusion to disease-relevant biology
rather than the entire Reactome catalog.

In [7]:
if config.USE_PATHWAY_FILTER:
    pathway_genes = build_disease_pathway_genes(
        "../../data/reactome/ReactomePathways.gmt", "../../data/reactome/reactome_relations.csv"
    )
    pd.DataFrame(sorted(pathway_genes), columns=["gene"]).to_csv(
        config.DISEASE_PATHWAY_GENES_FILE, index=False
    )
    print(f"[PATHWAY FILTER] {len(pathway_genes)} disease-pathway genes")

    before = len(non_drivers)
    non_drivers = apply_pathway_filter(non_drivers, pathway_genes)
    print(f"[PATHWAY FILTER] {len(non_drivers)} remaining (-{before - len(non_drivers)})")

[PATHWAY FILTER] kept 782 disease-related pathways, skipped 2048 unrelated pathways
[PATHWAY FILTER] 2505 disease-pathway genes
[PATHWAY FILTER] 2421 remaining (-70)


## 5. Length- and expression-matched negative sampling

Rather than keeping every surviving candidate as a negative, each driver gene is matched to its
nearest unused candidate(s) in log-transformed, z-scored covariate space (gene length,
expression, and optionally mutation rate). This corrects for drivers tending to be longer and
more highly expressed than a random background gene.

In [8]:
if config.USE_MATCHED_NEGATIVE_SAMPLING:
    gene_lengths = compute_gene_lengths("../../data/gencode.v49.basic.annotation.gtf")
    gene_expression = load_gene_expression("../data/gene_median_expression.csv")
    mutation_rate = (
        load_gene_mutation_rate(config.MUTATION_FREQUENCY_FILE)
        if config.MATCH_ON_MUTATION_RATE
        else None
    )

    match_df = match_negatives(
        positive_genes=driver_genes,
        candidate_negative_genes=non_drivers,
        gene_lengths=gene_lengths,
        gene_expression=gene_expression,
        mutation_rate=mutation_rate,
        n_per_positive=config.NEGATIVES_PER_POSITIVE,
        random_seed=config.RANDOM_SEED,
    )

    matched_negatives = set(match_df["matched_negative_gene"])
    print(
        f"[MATCHED NEGATIVES] {len(matched_negatives)} negatives matched to "
        f"{len(driver_genes)} positives"
    )

    match_df.to_csv(config.PROCESSED_DIR / "negative_gene_matches.csv", index=False)
    match_df.head()

[COVARIATES] dropped 16/763 genes missing a covariate value
[COVARIATES] dropped 947/2421 genes missing a covariate value
[MATCHING] warning: only found 0/1 unused matches for 'GAS7' — negative candidate pool may be too small or too dissimilar in this region of covariate space
[MATCHING] warning: only found 0/1 unused matches for 'RAD50' — negative candidate pool may be too small or too dissimilar in this region of covariate space
[MATCHING] warning: only found 0/1 unused matches for 'RAP1GDS1' — negative candidate pool may be too small or too dissimilar in this region of covariate space
[MATCHING] warning: only found 0/1 unused matches for 'NOTCH2' — negative candidate pool may be too small or too dissimilar in this region of covariate space
[MATCHING] warning: only found 0/1 unused matches for 'NSD3' — negative candidate pool may be too small or too dissimilar in this region of covariate space
[MATCHING] warning: only found 0/1 unused matches for 'EXT2' — negative candidate pool may 

### Check matching quality

Compares median length/expression(/mutation rate) across positives, the matched negatives, and the full unmatched candidate pool. Matched negatives should sit close to positives; the unmatched pool should not — that gap is the evidence the matching worked.

In [9]:
if config.USE_MATCHED_NEGATIVE_SAMPLING:
    pos_covariates = build_covariate_table(
        driver_genes, gene_lengths, gene_expression, mutation_rate
    )
    quality_report = summarize_matching_quality(
        pos_df=pos_covariates,
        matched_negative_genes=matched_negatives,
        unmatched_candidate_genes=non_drivers,
        gene_lengths=gene_lengths,
        gene_expression=gene_expression,
        mutation_rate=mutation_rate,
    )
    quality_report.to_csv(config.PROCESSED_DIR / "negative_matching_quality.csv", index=False)

    non_drivers = matched_negatives
    quality_report

[COVARIATES] dropped 16/763 genes missing a covariate value
[COVARIATES] dropped 947/2421 genes missing a covariate value


## 6. Build and save the final labeled gene table

In [10]:
labels_df = build_gene_labels(driver_genes, non_drivers)
print(labels_df["label"].value_counts())

labels_df.to_csv(config.GENE_LABELS_FILE, index=False)
print(f"[DONE] Saved -> {config.GENE_LABELS_FILE}")

driver_path = config.PROCESSED_DIR / "driver_genes.txt"
with open(driver_path, "w") as f:
    for gene in driver_genes:
        f.write(f"{gene}\n")

nondriver_path = config.PROCESSED_DIR / "non_driver_genes.txt"
with open(nondriver_path, "w") as f:
    for gene in non_drivers:
        f.write(f"{gene}\n")

print(f"[DONE] Saved driver genes -> {driver_path}")
print(f"[DONE] Saved non-driver genes -> {nondriver_path}")
labels_df.head()

label
1    763
0    470
Name: count, dtype: int64
[DONE] Saved -> ../data/processed/gene_labels_driver_vs_nondriver.csv
[DONE] Saved driver genes -> ../data/processed/driver_genes.txt
[DONE] Saved non-driver genes -> ../data/processed/non_driver_genes.txt


,gene,label
0,PRKCB,1
1,MAP2K2,1
2,SH3GL1,1
3,SMARCA4,1
4,FANCC,1


In [11]:
import os
import random
import sys
from pathlib import Path

import pandas as pd

# Run from the pipeline root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import config
from src.driver_labeling import (
    apply_mutation_frequency_filter,
    apply_pathway_filter,
    build_gene_labels,
    load_driver_genes,
)
from src.gene_universe import build_protein_coding_gene_universe
from src.negative_sampling import build_driver_exclusion_set
from src.pathway_filter import build_disease_pathway_genes


# Configuration for this run: retain all filtered negative candidates.
USE_MATCHED_NEGATIVE_SAMPLING = False

random.seed(config.RANDOM_SEED)
config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Protein-coding gene universe
all_genes = build_protein_coding_gene_universe(
    "../../data/gencode.v49.basic.annotation.gtf"
)
pd.DataFrame(sorted(all_genes), columns=["gene_name"]).to_csv(
    config.GENCODE_GENES_FILE, index=False
)
print(f"[GENE UNIVERSE] {len(all_genes)} protein-coding genes")

# 2. Positive drivers and expanded driver exclusion set
driver_genes = load_driver_genes("../../data/Census_allWed.tsv")
print(f"[DRIVERS] {len(driver_genes)} CGC driver genes")

driver_exclusion_set = build_driver_exclusion_set(
    ncg_file=config.NCG_FILE if config.NCG_FILE.exists() else None,
    cgc_file=config.CGC_CENSUS_FILE if config.CGC_CENSUS_FILE.exists() else None,
    intogen_file=config.INTOGEN_FILE if config.INTOGEN_FILE.exists() else None,
    bailey_file=config.BAILEY_FILE if config.BAILEY_FILE.exists() else None,
)

non_drivers = all_genes - driver_exclusion_set
print(f"[NON-DRIVERS] {len(non_drivers)} raw candidates")

# 3. Remove frequently mutated genes
if config.USE_MUTATION_FILTER:
    before = len(non_drivers)
    non_drivers = apply_mutation_frequency_filter(
        non_drivers,
        config.MUTATION_FREQUENCY_FILE,
        threshold=config.MUTATION_FREQUENCY_THRESHOLD,
    )
    print(
        f"[MUTATION FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )

# 4. Remove disease-pathway genes
if config.USE_PATHWAY_FILTER:
    pathway_genes = build_disease_pathway_genes(
        "../../data/reactome/ReactomePathways.gmt",
        "../../data/reactome/reactome_relations.csv",
    )
    pd.DataFrame(sorted(pathway_genes), columns=["gene"]).to_csv(
        config.DISEASE_PATHWAY_GENES_FILE, index=False
    )

    before = len(non_drivers)
    non_drivers = apply_pathway_filter(non_drivers, pathway_genes)
    print(
        f"[PATHWAY FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )

# Keep all filtered negatives; do not covariate-match.
print(f"[FINAL NEGATIVES] {len(non_drivers)} candidates retained")

# 5. Build and save labels
labels_df = build_gene_labels(driver_genes, non_drivers)
print(labels_df["label"].value_counts())

labels_df.to_csv(config.GENE_LABELS_FILE, index=False)

driver_path = config.PROCESSED_DIR / "driver_genes.txt"
nondriver_path = config.PROCESSED_DIR / "non_driver_genes.txt"

driver_path.write_text("\n".join(sorted(driver_genes)) + "\n")
nondriver_path.write_text("\n".join(sorted(non_drivers)) + "\n")

print(f"[DONE] Labels: {config.GENE_LABELS_FILE}")
print(f"[DONE] Drivers: {driver_path}")
print(f"[DONE] Non-drivers: {nondriver_path}")

labels_df.head()

[GENE UNIVERSE] 20070 protein-coding genes
[DRIVERS] 763 CGC driver genes
[NCG] skipped (no file provided)
[CGC] 763 genes
[IntOGen] skipped (no file provided)
[Bailey et al.] skipped (no file provided)
[DRIVER EXCLUSION SET] 763 unique genes across all sources
[NON-DRIVERS] 19326 raw candidates
[MUTATION FILTER] 2491 remaining (-16835)
[PATHWAY FILTER] kept 782 disease-related pathways, skipped 2048 unrelated pathways
[PATHWAY FILTER] 2421 remaining (-70)
[FINAL NEGATIVES] 2421 candidates retained
label
0    2421
1     763
Name: count, dtype: int64
[DONE] Labels: ../data/processed/gene_labels_driver_vs_nondriver.csv
[DONE] Drivers: ../data/processed/driver_genes.txt
[DONE] Non-drivers: ../data/processed/non_driver_genes.txt


,gene,label
0,PRKCB,1
1,MAP2K2,1
2,SH3GL1,1
3,SMARCA4,1
4,FANCC,1


In [12]:
driver_path

PosixPath('../data/processed/driver_genes.txt')